# Week 08 Portfolio: Positional-Encoding Ablation for 24 h-Ahead Feeder-Load Forecasting
*CSP Machine Learning Engineering*

**Objective: A controlled positional-encoding ablation (4 arms x 5 seeds) on one GPU, with latency, memory and seed spread logged.**

**Question:** for a small Transformer that forecasts rural co-op demand, does the positional-encoding choice
matter more than run-to-run noise?
**This script's job:** generate the dataset, train every (arm, seed) with one factor changing, report means with
seed-level spread, log wall-clock latency and peak memory beside accuracy, and write tagged results.

| Arm | What it is |
|---|---|
| none | no positional information (control: the model cannot tell hours apart) |
| sinusoidal | fixed sine/cosine table added to the input embedding (reference arm) |
| learned | trainable position embedding added to the input embedding (+10,752 parameters) |
| rope | rotary encoding applied to queries and keys inside attention |

The data are synthetic and generated here from a fixed seed (8 feeders x 540 days, hourly). Everything except
`pos_enc` is held fixed: data, split, model size, optimiser, schedule, epochs, attention code, checkpoint rule.
Results are written to `results/*_<tag>.*`; the tag is `t4gpu` on a Colab T4 and `cpu` elsewhere, so a CPU run
never overwrites T4 results. Set `ABLATE_QUICK=1` for a smoke test (outputs tagged `_quick`, not meaningful).

Rerunning with `results/runs_<tag>.csv` present resumes from the saved runs and only regenerates statistics,
tables and plots; set `ABLATE_FRESH=1` to retrain everything.

**References**

- Su, J., Lu, Y., Pan, S., Murtadha, A., Wen, B., & Liu, Y. (2024). RoFormer: Enhanced transformer with rotary position embedding. Neurocomputing, 568, 127063. https://doi.org/10.1016/j.neucom.2023.127063
- Vaswani, A., Shazeer, N., Parmar, N., Uszkoreit, J., Jones, L., Gomez, A. N., Kaiser, L., & Polosukhin, I. (2017). Attention is all you need. Advances in Neural Information Processing Systems, 30. https://arxiv.org/abs/1706.03762
- Holm, S. (1979). A simple sequentially rejective multiple test procedure. Scandinavian Journal of Statistics, 6(2), 65-70. https://www.jstor.org/stable/4615733
- Welch, B. L. (1947). The generalization of "Student's" problem when several different population variances are involved. Biometrika, 34(1-2), 28-35. https://doi.org/10.1093/biomet/34.1-2.28
- European Parliament and Council. (2024). Regulation (EU) 2024/1689 (Artificial Intelligence Act), Annex IV. https://eur-lex.europa.eu/eli/reg/2024/1689/oj
- National Institute of Standards and Technology. (2023). Artificial Intelligence Risk Management Framework (AI RMF 1.0) (NIST AI 100-1). https://doi.org/10.6028/NIST.AI.100-1

## Colab Bootstrap

A Colab runtime starts with only this notebook. This cell clones the repo once and changes into it, so `scripts/` and `results/` exist. Running it again does nothing. A private repo clones with the `GITHUB_TOKEN` secret.

In [ ]:
# Colab bootstrap: clone the repo once and move into it. Safe to re-run: skipped inside the clone.
import base64, os, subprocess

REPO_URL = "https://github.com/davisp8-creator/Week-08-Career-Portfolio.git"
REPO_DIR = REPO_URL.rsplit("/", 1)[-1].removesuffix(".git")

def colab_token():
    try:
        from google.colab import userdata
        return userdata.get("GITHUB_TOKEN") or None
    except Exception:  # not on Colab, secret missing, or notebook access not granted
        return None

if not os.path.exists("env_setup.py"):
    if not os.path.isdir(REPO_DIR):
        cmd = ["git"]
        token = colab_token()  # only needed when the repo is private; sent as a header, never stored
        if token:
            basic = base64.b64encode(f"x-access-token:{token}".encode()).decode()
            cmd += ["-c", f"http.https://github.com/.extraheader=AUTHORIZATION: basic {basic}"]
        subprocess.run(cmd + ["clone", "-q", REPO_URL], check=True)
    os.chdir(REPO_DIR)
print("working directory:", os.getcwd())

## Run Settings

`RETRAIN = True` retrains all 20 (arm, seed) runs on this machine, about 8 minutes on a T4, so `results/runs_<tag>.csv` and `manifest_<tag>.json` come from the same session. The committed T4 results from the first session are replaced in your clone (they stay in git history). Set it to `False` only to regenerate tables from rows already in `results/`; the manifest would then record this session's hardware beside older runs.

In [ ]:
import os

RETRAIN = True
os.environ["ABLATE_FRESH"] = "1" if RETRAIN else "0"

## Step 1: Environment Setup

Detects Colab vs local, confirms a T4 (via `nvidia-smi` and `torch.cuda`), tells a virtual machine from a
physical PC, installs only missing packages (never Colab's CUDA torch), and warns if the runtime is CPU-only.
The same code ships as `env_setup.py` for local runs and the installers.

References: [1](https://research.google.com/colaboratory/faq.html) · [2](https://docs.nvidia.com/deploy/nvidia-smi/index.html) · [3](https://docs.pytorch.org/docs/stable/generated/torch.cuda.is_available.html)

In [ ]:
# ── env_setup.py (inlined by scripts/build_notebook.py; edit the .py, then rebuild) ──
import importlib
import importlib.util
import json
import os
import platform
import re
import shutil
import subprocess
import sys
from pathlib import Path

# import name -> pip spec. Every third-party import in the assignment script must appear here.
PACKAGE_MAP = {
    "numpy": "numpy",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "torch": "torch",
    "scipy": "scipy",
    "yaml": "pyyaml",
    "psutil": "psutil",
}
ASSIGNMENT_SCRIPT = Path("scripts") / "run_ablation.py"
# helper modules the assignment script imports at run time; their third-party imports count too
HELPER_SCRIPTS = [Path("scripts") / n for n in ("power_analysis.py", "repeat_study.py", "update_readme.py")]
LOCAL_MODULES = {"env_setup", "update_readme"}
COLAB_ONLY_MODULES = {"google"}   # google.colab: provided by Colab, only imported behind a platform check
MIN_PYTHON = (3, 10)
T4_RUNTIME_HINT = (
    "In Colab: Runtime > Change runtime type > Hardware accelerator: T4 GPU > Save,\n"
    "then Runtime > Run all. (The runtime restarts, so re-run from the top.)"
)
HOST_LABEL = {
    "colab-t4": "Colab (T4 GPU)", "colab-gpu": "Colab (other GPU)", "colab-cpu": "Colab (CPU runtime)",
    "intel-vm": "Intel VM (CPU only)", "intel-host": "Intel PC (CPU only)", "apple-silicon": "Apple silicon (CPU only)",
}


# ── detection ────────────────────────────────────────────────────────────────
def is_colab() -> bool:
    if importlib.util.find_spec("google") is not None:
        try:
            import google.colab  # noqa: F401
            return True
        except ImportError:
            pass
    return "COLAB_RELEASE_TAG" in os.environ or "COLAB_GPU" in os.environ


def query_nvidia_smi() -> dict | None:
    """GPU name / VRAM / driver from nvidia-smi: the torch-independent signal that a GPU is attached."""
    if shutil.which("nvidia-smi") is None:
        return None
    out = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader,nounits"],
        capture_output=True, text=True, check=False)
    if out.returncode != 0 or not out.stdout.strip():
        return None
    name, mem_mib, driver = [s.strip() for s in out.stdout.strip().splitlines()[0].split(",")]
    return {"name": name, "vram_gb": round(float(mem_mib) / 1024, 1), "driver": driver}


def _run(cmd: list[str]) -> str:
    try:
        return subprocess.run(cmd, capture_output=True, text=True, timeout=30, check=False).stdout.strip()
    except (OSError, subprocess.SubprocessError):
        return ""


def _powershell(expr: str) -> str:
    return _run(["powershell", "-NoProfile", "-Command", expr])


def _read(path: str) -> str:
    try:
        return Path(path).read_text(errors="ignore").strip()
    except OSError:
        return ""


VM_WORDS = ("virtual", "vmware", "kvm", "qemu", "xen", "hyper-v", "google compute", "bochs", "parallels", "vbox")


def detect_cpu_and_vm() -> dict:
    """CPU model/vendor and whether this machine is a virtual machine, without needing torch.

    Windows: Win32_ComputerSystem / Win32_Processor.  Linux: DMI + /proc/cpuinfo.  macOS: sysctl.
    """
    system = platform.system()
    cpu, product, vendor_sys, hv_flag = "", "", "", None
    if system == "Windows":
        cpu = _powershell("(Get-CimInstance Win32_Processor | Select-Object -First 1).Name")
        csys = _powershell("$c = Get-CimInstance Win32_ComputerSystem; \"$($c.Manufacturer)|$($c.Model)|$($c.HypervisorPresent)\"")
        vendor_sys, product, hv = (csys.split("|") + ["", "", ""])[:3]
        hv_flag = hv.strip().lower() == "true"
    elif system == "Linux":
        info = _read("/proc/cpuinfo")
        m = re.search(r"model name\s*:\s*(.+)", info)
        cpu = m.group(1).strip() if m else ""
        product, vendor_sys = _read("/sys/class/dmi/id/product_name"), _read("/sys/class/dmi/id/sys_vendor")
        hv_flag = bool(re.search(r"^flags\s*:.*\bhypervisor\b", info, flags=re.M))
        if shutil.which("systemd-detect-virt"):
            sdv = _run(["systemd-detect-virt"])
            if sdv and sdv != "none":
                product = f"{product} ({sdv})".strip()
    elif system == "Darwin":
        cpu = _run(["sysctl", "-n", "machdep.cpu.brand_string"])
        hv_flag = _run(["sysctl", "-n", "kern.hv_vmm_present"]) == "1"
        product, vendor_sys = _run(["sysctl", "-n", "hw.model"]), "Apple"
    cpu = cpu or platform.processor() or platform.machine()
    sig = f"{product} {vendor_sys}".lower()
    # Model/vendor strings identify a VM reliably. HypervisorPresent alone is NOT enough on Windows:
    # a physical host running Hyper-V/VBS also reports it, so it only counts when the model agrees
    # or (macOS/Linux) the kernel flag is the sole signal available.
    virtualized = any(w in sig for w in VM_WORDS) or (system != "Windows" and bool(hv_flag))
    low = cpu.lower()
    vendor = "intel" if ("intel" in low or "genuineintel" in low) else "apple" if ("apple" in low or platform.machine() == "arm64") \
        else "amd" if "amd" in low else "other"
    return {"cpu_model": cpu, "cpu_vendor": vendor, "virtualized": virtualized, "vm_product": product or None,
            "vm_vendor": vendor_sys or None, "hypervisor_flag": hv_flag}


def ram_gib() -> float | None:
    try:
        import psutil
        return round(psutil.virtual_memory().total / 2**30, 1)
    except ImportError:
        return None


def classify_host(env: dict, gpu_name: str | None = None, device: str = "cpu") -> str:
    """One label that says what the numbers were measured on."""
    if env["platform"] == "colab":
        smi = env["nvidia_smi"]
        name = gpu_name or (smi["name"] if smi else None)
        if name and "T4" in name:
            return "colab-t4"
        return "colab-gpu" if (name or device == "cuda") else "colab-cpu"
    if env["cpu_vendor"] == "apple":
        return "apple-silicon"
    if env["cpu_vendor"] == "intel":
        return "intel-vm" if env["virtualized"] else "intel-host"
    return "other-vm" if env["virtualized"] else "other-host"


def detect_environment() -> dict:
    colab = is_colab()
    system = platform.system()  # "Windows" | "Darwin" | "Linux"
    env = {
        "platform": "colab" if colab else {"Windows": "windows", "Darwin": "macos"}.get(system, "linux"),
        "system": system,
        "python": platform.python_version(),
        "nvidia_smi": query_nvidia_smi(),
    }
    env.update(detect_cpu_and_vm())
    env["host_class"] = classify_host(env)
    return env


# ── installation ─────────────────────────────────────────────────────────────
def required_imports(script: Path) -> list[str]:
    """Top-level third-party imports of the assignment script (stdlib and local modules excluded)."""
    import ast
    lines = script.read_text(encoding="utf-8").splitlines()
    source = "\n".join("" if ln.lstrip()[:1] in ("!", "%") else ln for ln in lines)  # drop notebook magics
    found = set()
    for node in ast.walk(ast.parse(source)):
        if isinstance(node, ast.Import):
            found.update(a.name.split(".")[0] for a in node.names)
        elif isinstance(node, ast.ImportFrom) and node.module and node.level == 0:
            found.add(node.module.split(".")[0])
    return sorted(found - set(sys.stdlib_module_names) - LOCAL_MODULES - COLAB_ONLY_MODULES)


def missing_packages(imports: list[str]) -> list[str]:
    unmapped = [i for i in imports if i not in PACKAGE_MAP]
    if unmapped:
        raise SystemExit(f"Unmapped third-party import(s): {unmapped}. Add them to PACKAGE_MAP in env_setup.py.")
    return [PACKAGE_MAP[i] for i in imports if importlib.util.find_spec(i) is None]


def project_root() -> Path:
    # __file__ is absent when this module is inlined into the Colab notebook
    return Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()


def install_missing(env: dict, script: Path | None = None) -> list[str]:
    script = script or project_root() / ASSIGNMENT_SCRIPT
    if script.exists():
        found = set(required_imports(script))
        for helper in HELPER_SCRIPTS:
            if (script.parent.parent / helper).exists():
                found |= set(required_imports(script.parent.parent / helper))
        imports = sorted(found)
    else:
        imports = list(PACKAGE_MAP)
    todo = missing_packages(imports)
    if env["platform"] == "colab" and "torch" in todo:
        # Colab ships a CUDA build of torch; this only triggers if someone uninstalled it.
        print("[setup] torch is missing on Colab; installing the default (CUDA-enabled on Linux) build.")
    if todo and env["platform"] != "colab" and os.environ.get("ALLOW_UNPINNED_INSTALL") != "1":
        raise SystemExit(f"[setup] missing packages: {' '.join(todo)}. Install the pinned set first "
                         "(requirements-cpu.txt or requirements-t4.txt, plus the matching torch build named in it), "
                         "or set ALLOW_UNPINNED_INSTALL=1 to accept whatever pip resolves; the manifest will record it.")
    if todo:
        print(f"[setup] installing missing packages: {' '.join(todo)}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *todo])
        importlib.invalidate_caches()
    else:
        print(f"[setup] all required packages already present: {' '.join(PACKAGE_MAP[i] for i in imports)}")
    return todo


# ── device resolution ────────────────────────────────────────────────────────
def resolve_device(env: dict) -> dict:
    torch = importlib.import_module("torch")
    info = {"torch": torch.__version__, "torch_cuda_build": torch.version.cuda,
            "device": "cpu", "gpu_name": None, "vram_gb": None, "compute_capability": None,
            "mps_available": bool(getattr(torch.backends, "mps", None) and torch.backends.mps.is_available())}
    if torch.cuda.is_available():
        p = torch.cuda.get_device_properties(0)
        info.update(device="cuda", gpu_name=p.name, vram_gb=round(p.total_memory / 2**30, 1),
                    compute_capability=f"{p.major}.{p.minor}")
    info["is_t4"] = bool(info["gpu_name"] and "T4" in info["gpu_name"])
    info["tag"] = "t4gpu" if info["is_t4"] else ("gpu" if info["device"] == "cuda" else "cpu")
    info["host_class"] = classify_host(env, info["gpu_name"], info["device"])
    return info


def _warn(msg: str) -> None:
    bar = "!" * 72
    print(f"\n{bar}\n{msg}\n{bar}\n")


def diagnose(env: dict) -> None:
    """Explain anything that would make the run slower or the results mislabelled."""
    smi = env["nvidia_smi"]
    if env["platform"] == "colab" and smi is None:
        _warn("Colab is on a CPU-only runtime. Everything still runs, but GPU rows fall back to the CPU,\n"
              "so timings and peak VRAM will not be T4 numbers.\n" + T4_RUNTIME_HINT)
    elif smi and env["device"] == "cpu":
        _warn(f"nvidia-smi sees a GPU ({smi['name']}) but torch cannot use it "
              f"(torch {env['torch']}, CUDA build: {env['torch_cuda_build']}).\n"
              "Usually a CPU-only torch was pip-installed over the CUDA build (for example, from\n"
              "requirements.txt). On Colab: Runtime > Disconnect and delete runtime, then run again\n"
              "without installing torch. Locally: install the CUDA build from pytorch.org.")
    elif env["device"] == "cuda" and not env["is_t4"]:
        print(f"[setup] note: GPU is {env['gpu_name']}, not a T4. Results are tagged '{env['tag']}' "
              "so they are not confused with the graded T4 run.")
    if env["platform"] != "colab" and env["device"] == "cpu" and env["virtualized"]:
        print(f"[setup] note: running inside a virtual machine ({env['vm_product']}). Latency here includes "
              "hypervisor scheduling jitter; latency is reported as p50, p95 and the spread between interleaved rounds.")
    if env["mps_available"] and env["device"] == "cpu":
        print("[setup] note: Apple MPS is available but deliberately not used: the ablation times every arm on one "
              "reference device per tag, and MPS gives no determinism guarantee for these ops.")


def hardware_snapshot(env: dict) -> dict:
    """Everything a reader needs to interpret wall-clock and VRAM numbers."""
    snap = {k: env.get(k) for k in ("platform", "system", "python", "torch", "torch_cuda_build",
                                    "device", "gpu_name", "vram_gb", "compute_capability", "tag", "host_class",
                                    "cpu_model", "cpu_vendor", "virtualized", "vm_product")}
    snap["host_label"] = HOST_LABEL.get(env.get("host_class"), env.get("host_class"))
    snap["cpu_count"] = os.cpu_count()
    snap["ram_gib"] = ram_gib()
    snap["driver"] = (env.get("nvidia_smi") or {}).get("driver")
    for pkg in ("numpy", "pandas", "matplotlib", "scipy", "yaml"):
        try:
            snap[pkg] = importlib.import_module(pkg).__version__
        except ImportError:
            snap[pkg] = None
    return snap


def setup(install: bool = True, require_t4: bool = False) -> dict:
    """Detect the environment, install missing packages, resolve the device, and print a summary."""
    if sys.version_info < MIN_PYTHON:
        raise SystemExit(f"Python {MIN_PYTHON[0]}.{MIN_PYTHON[1]}+ required, found {platform.python_version()}")
    env = detect_environment()
    if install:
        install_missing(env)
    env.update(resolve_device(env))
    diagnose(env)
    if require_t4 and not env["is_t4"]:
        raise RuntimeError("require_t4=True but no Tesla T4 is visible.\n" + T4_RUNTIME_HINT)
    gpu = f"{env['gpu_name']} ({env['vram_gb']} GB, sm_{(env['compute_capability'] or '').replace('.', '')})" \
        if env["device"] == "cuda" else "none"
    print(f"[setup] host={env['host_class']} ({HOST_LABEL.get(env['host_class'], env['host_class'])})  "
          f"platform={env['platform']}  python={env['python']}  torch={env['torch']}  device={env['device']}  gpu={gpu}  "
          f"results tag='{env['tag']}'")
    print(f"[setup] cpu={env['cpu_model']}  virtualized={env['virtualized']}"
          + (f" ({env['vm_product']})" if env["vm_product"] and env["virtualized"] else ""))
    return env
# ── end env_setup.py ──

ROOT = Path.cwd()

ENV = setup(require_t4=False)  # set require_t4=True to refuse anything but a Colab T4
if not (ROOT / ".git").exists():
    pass

## Step 2: Imports, Single-Source Constants & Hardware-Tagged Output Paths

Every budget lives in configs/ablation.yaml. `ABLATE_QUICK=1` applies its `quick:` block, so the smoke test and
the full run differ only there. Outputs are tagged by hardware (`t4gpu` / `cpu`, plus `_quick` for smoke tests).

References: [1](https://docs.python.org/3/library/time.html#time.perf_counter) · [2](https://docs.pytorch.org/docs/stable/notes/randomness.html) · [3](https://docs.pytorch.org/docs/stable/generated/torch.use_deterministic_algorithms.html)

In [ ]:
# Install required libraries: run the installer for your OS (or `python env_setup.py`);
# on Colab, Step 1 installs anything missing and leaves the CUDA torch build alone.
import os

os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")  # deterministic cuBLAS; set before CUDA initialises

import hashlib
import json
import math
import random
import time

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import psutil
import torch
import torch.nn as nn
import torch.nn.functional as F
import yaml
from scipy import stats

T_START = time.perf_counter()
IN_NOTEBOOK = "ipykernel" in sys.modules
QUICK = os.environ.get("ABLATE_QUICK") == "1"  # smoke test only: results NOT meaningful
FRESH = os.environ.get("ABLATE_FRESH") == "1"  # ignore saved runs and retrain everything


def deep_merge(base, over):
    out = dict(base)
    for k, v in over.items():
        out[k] = deep_merge(out[k], v) if isinstance(v, dict) and isinstance(out.get(k), dict) else v
    return out


if not (ROOT / "configs" / "ablation.yaml").exists():
    raise FileNotFoundError(f"{ROOT / 'configs' / 'ablation.yaml'} not found. The working directory must be the repo root "
                            "(run from a clone of this repository).")
CFG = yaml.safe_load((ROOT / "configs" / "ablation.yaml").read_text(encoding="utf-8-sig"))
CFG = deep_merge({k: v for k, v in CFG.items() if k != "quick"}, CFG["quick"]) if QUICK else \
    {k: v for k, v in CFG.items() if k != "quick"}
ARMS, SEEDS, REF = CFG["arms"], CFG["seeds"], CFG["reference_arm"]
DEVICE = torch.device(ENV["device"])
TAG = ENV["tag"] + ("_quick" if QUICK else "")
RESULTS, DATA_DIR = ROOT / "results", ROOT / "data"
RESULTS.mkdir(exist_ok=True)
torch.set_num_threads(CFG["threads"])
torch.use_deterministic_algorithms(True, warn_only=True)
print(f"arms={ARMS} seeds={SEEDS} tag={TAG} device={DEVICE} quick={QUICK}")

## Step 3: Synthetic Demand Data

Hourly load (kW) and temperature for rural co-op feeders: daily residential/commercial shapes, weekday and
holiday factors, temperature response around 15 C, summer-night irrigation, 4% multiplicative noise and rare
outages that are left in on purpose. Fully determined by `data_seed`; the SHA-256 of the CSV is recorded so a
reader can verify the rerun produced the same bytes. These are our generator's assumptions, not measurements.

References: [1](https://numpy.org/doc/stable/reference/random/generator.html) · [2](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.to_csv.html)

In [ ]:
def generate_demand(dc):
    rng = np.random.default_rng(dc["data_seed"])
    n_hours = dc["n_days"] * 24
    t = np.arange(n_hours)
    idx = pd.Timestamp(dc["start"]) + pd.to_timedelta(t, unit="h")
    hod, dow = idx.hour.values, idx.day_of_week.values
    annual = 12.0 - 11.0 * np.cos(2 * np.pi * (t - 400) / 8760)
    diurnal = 4.0 * np.sin(2 * np.pi * (t % 24 - 9) / 24)
    eps = rng.normal(0, 1.0, n_hours)
    ar = np.zeros(n_hours)
    for i in range(1, n_hours):  # AR(1) weather noise with day-scale persistence
        ar[i] = 0.97 * ar[i - 1] + eps[i]
    temp = annual + diurnal + 0.9 * ar
    holidays = set(rng.choice(dc["n_days"], size=dc["n_days"] // 40, replace=False).tolist())
    is_hol = np.array([(d // 24) in holidays for d in t])
    frames = []
    for f in range(dc["n_feeders"]):
        base = rng.uniform(80, 400)
        res_share = rng.uniform(0.3, 0.9)
        phase = rng.normal(0, 1.2)
        res = (0.6 + 0.5 * np.exp(-0.5 * ((hod - 7.5 - phase) / 1.6) ** 2)
               + 0.9 * np.exp(-0.5 * ((hod - 19 - phase) / 2.2) ** 2))
        com = 0.5 + 0.8 / (1 + np.exp(-(hod - 7.5))) - 0.8 / (1 + np.exp(-(hod - 17.5)))
        shape = res_share * res + (1 - res_share) * com
        weekly = np.where(dow >= 5, rng.uniform(0.82, 0.95), 1.0)
        weekly = np.where(is_hol, 0.85, weekly)
        ft = temp + rng.normal(0, 0.8)
        climate = (1 + rng.uniform(0.012, 0.03) * np.maximum(15 - ft, 0)
                   + rng.uniform(0.004, 0.015) * np.maximum(ft - 22, 0))
        irrigation = 1 + rng.uniform(0.0, 0.3) * ((idx.month.values >= 6) & (idx.month.values <= 8) & (hod >= 22) | (hod <= 4))
        y = base * shape * weekly * climate * irrigation * (1 + 0.04 * rng.normal(size=n_hours))
        for s in rng.choice(n_hours - 12, size=max(1, n_hours // 6000), replace=False):  # rare outages
            y[s:s + rng.integers(2, 8)] *= rng.uniform(0.0, 0.2)
        frames.append(pd.DataFrame({"feeder": f, "timestamp": idx, "load_kw": y.round(3), "temp_c": ft.round(2)}))
    return pd.concat(frames, ignore_index=True)


def build_dataset(dc, out_dir):
    out_dir.mkdir(parents=True, exist_ok=True)
    path = out_dir / ("demand_quick.csv" if QUICK else "demand.csv")
    df = generate_demand(dc)
    df.to_csv(path, index=False, lineterminator="\n")
    sha = hashlib.sha256(path.read_bytes()).hexdigest()
    path.with_suffix(".meta.json").write_text(json.dumps({"sha256": sha, "rows": len(df), "config": dc}, indent=2))
    return df, sha


DF, DATA_SHA = build_dataset(CFG["data"], DATA_DIR)
print(f"data: {len(DF):,} rows, {CFG['data']['n_feeders']} feeders, sha256={DATA_SHA[:16]}...")

## Step 4: Windows and Chronological Splits

168 h lookback -> 24 h horizon. Each feeder is split chronologically 70/15/15; normalisation uses the TRAIN
segment only; no window crosses a split boundary. Two no-learning references are scored on the test set:
"same hours yesterday" and "same hours one week earlier".

References: [1](https://otexts.com/fpp3/tspred.html) · [2](https://scikit-learn.org/stable/common_pitfalls.html#data-leakage)

In [ ]:
def make_windows(df, cfg):
    L, H = cfg["window"]["lookback"], cfg["window"]["horizon"]
    out = {k: ([], [], []) for k in ("train", "val", "test")}  # X, Y, load std in kW
    for _, g in df.groupby("feeder"):
        v = g.sort_values("timestamp")[["load_kw", "temp_c"]].to_numpy(np.float32)
        n = len(v)
        a, b = int(n * cfg["split"]["train"]), int(n * (cfg["split"]["train"] + cfg["split"]["val"]))
        mu, sd = v[:a].mean(0), v[:a].std(0) + 1e-6
        z = (v - mu) / sd
        for name, (lo, hi), stride in (("train", (0, a), cfg["window"]["stride_train"]),
                                       ("val", (a, b), cfg["window"]["stride_eval"]),
                                       ("test", (b, n), cfg["window"]["stride_eval"])):
            for s in range(lo, hi - L - H + 1, stride):
                out[name][0].append(z[s:s + L])
                out[name][1].append(z[s + L:s + L + H, 0])
                out[name][2].append(sd[0])
    return {k: tuple(torch.tensor(np.array(x), dtype=torch.float32) for x in v) for k, v in out.items()}


def naive_baselines(X, Y, S):
    day, week = X[:, -24:, 0], X[:, :24, 0]  # lookback is exactly 168 h, so index k is one week before target hour k
    return {"naive_day_mae_kw": ((day - Y).abs() * S[:, None]).mean().item(),
            "naive_week_mae_kw": ((week - Y).abs() * S[:, None]).mean().item()}


W = make_windows(DF, CFG)
NAIVE = naive_baselines(*W["test"])
print({k: tuple(v[0].shape) for k, v in W.items()}, {k: round(v, 2) for k, v in NAIVE.items()})

## Step 5: Forecaster with Four Positional Encodings, and Reference Checks

One Transformer encoder with a last-token readout. The ONLY difference between arms is `pos_enc`. Attention is
written out (softmax(QK^T)V) so CUDA runs stay deterministic. Two checks: the hand-built attention must match
F.scaled_dot_product_attention, and RoPE scores must depend only on the offset between positions.

References: [1](https://arxiv.org/abs/2104.09864) · [2](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.scaled_dot_product_attention.html)

In [ ]:
def sinusoidal_table(length, d):
    pos = torch.arange(length).unsqueeze(1)
    div = torch.exp(torch.arange(0, d, 2) * (-math.log(10000.0) / d))
    pe = torch.zeros(length, d)
    pe[:, 0::2], pe[:, 1::2] = torch.sin(pos * div), torch.cos(pos * div)
    return pe


def rope_tables(length, head_dim):
    inv = 1.0 / (10000 ** (torch.arange(0, head_dim, 2).float() / head_dim))
    ang = torch.arange(length).float().unsqueeze(1) * inv.unsqueeze(0)
    return ang.cos(), ang.sin()


def apply_rope(x, cos, sin):  # x: batch, heads, length, head_dim
    x1, x2 = x[..., 0::2], x[..., 1::2]
    out = torch.empty_like(x)
    out[..., 0::2], out[..., 1::2] = x1 * cos - x2 * sin, x1 * sin + x2 * cos
    return out


class Block(nn.Module):
    def __init__(self, d, heads, ff, dropout, rope):
        super().__init__()
        self.h, self.hd, self.rope = heads, d // heads, rope
        self.qkv, self.proj = nn.Linear(d, 3 * d), nn.Linear(d, d)
        self.n1, self.n2 = nn.LayerNorm(d), nn.LayerNorm(d)
        self.ff = nn.Sequential(nn.Linear(d, ff), nn.GELU(), nn.Linear(ff, d))
        self.drop = nn.Dropout(dropout)

    def attend(self, x, cos, sin):
        B, L, D = x.shape
        q, k, v = self.qkv(self.n1(x)).view(B, L, 3, self.h, self.hd).permute(2, 0, 3, 1, 4)
        if self.rope:
            q, k = apply_rope(q, cos, sin), apply_rope(k, cos, sin)
        att = torch.softmax(q @ k.transpose(-2, -1) / math.sqrt(self.hd), dim=-1)
        return (att @ v).transpose(1, 2).reshape(B, L, D), (q, k, v)

    def forward(self, x, cos, sin):
        y, _ = self.attend(x, cos, sin)
        x = x + self.drop(self.proj(y))
        return x + self.drop(self.ff(self.n2(x)))


class Forecaster(nn.Module):
    def __init__(self, pos_enc, n_in, lookback, horizon, d=64, heads=4, layers=2, ff=128, dropout=0.1):
        super().__init__()
        assert pos_enc in {"none", "sinusoidal", "learned", "rope"}
        self.pos_enc = pos_enc
        self.inp = nn.Linear(n_in, d)
        if pos_enc == "sinusoidal":
            self.register_buffer("pe", sinusoidal_table(lookback, d))
        elif pos_enc == "learned":
            self.pe = nn.Parameter(torch.randn(lookback, d) * 0.02)
        if pos_enc == "rope":
            c, s = rope_tables(lookback, d // heads)
            self.register_buffer("cos", c)
            self.register_buffer("sin", s)
        else:
            self.cos = self.sin = None
        self.blocks = nn.ModuleList(Block(d, heads, ff, dropout, pos_enc == "rope") for _ in range(layers))
        self.norm = nn.LayerNorm(d)
        self.head = nn.Linear(d, horizon)

    def forward(self, x):
        h = self.inp(x)
        if self.pos_enc in ("sinusoidal", "learned"):
            h = h + self.pe
        for b in self.blocks:
            h = b(h, self.cos, self.sin)
        return self.head(self.norm(h)[:, -1])  # last-token readout is position-sensitive


def build_model(arm):
    return Forecaster(arm, n_in=2, lookback=CFG["window"]["lookback"], horizon=CFG["window"]["horizon"], **CFG["model"])


def reference_checks():
    torch.manual_seed(0)
    blk = Block(32, 4, 64, 0.0, rope=False).eval()
    x = torch.randn(3, 20, 32)
    mine, (q, k, v) = blk.attend(x, None, None)
    ref = F.scaled_dot_product_attention(q, k, v).transpose(1, 2).reshape(3, 20, 32)
    attn_diff = (mine - ref).abs().max().item()
    c, s = rope_tables(40, 8)
    q0, k0 = torch.randn(1, 1, 1, 8), torch.randn(1, 1, 1, 8)

    def score(i, j):
        return (apply_rope(q0, c[i:i + 1], s[i:i + 1]) * apply_rope(k0, c[j:j + 1], s[j:j + 1])).sum().item()

    rope_diff = abs(score(5, 2) - score(25, 22))  # same offset (3), different absolute position
    print(f"reference check: hand-built attention vs F.scaled_dot_product_attention max|diff| = {attn_diff:.2e}")
    print(f"reference check: RoPE score at offset 3, positions (5,2) vs (25,22): |diff| = {rope_diff:.2e}")
    assert attn_diff < 1e-5 and rope_diff < 1e-5, "reference check failed"
    return {"attention_max_abs_diff": attn_diff, "rope_offset_invariance_diff": rope_diff}


CHECKS = reference_checks()
print({a: sum(p.numel() for p in build_model(a).parameters()) for a in ARMS})

## Step 6: Train and Evaluate One (Arm, Seed)

Same recipe for every arm: AdamW, one-cycle schedule, L1 loss, gradient clipping, best-validation-epoch
checkpoint. Batch order comes from a generator seeded by the seed alone, so seed k sees the same batches in
every arm. Test error is computed once, at the chosen checkpoint, in kW. Peak GPU memory is reported above the
allocation present before the model loaded (library workspace is not charged to the arm).

References: [1](https://docs.pytorch.org/docs/stable/generated/torch.optim.lr_scheduler.OneCycleLR.html) · [2](https://docs.pytorch.org/docs/stable/generated/torch.cuda.max_memory_allocated.html)

In [ ]:
def set_seed(s):
    random.seed(s)
    np.random.seed(s)
    torch.manual_seed(s)
    torch.cuda.manual_seed_all(s)


@torch.no_grad()
def evaluate(model, X, Y, S, bs=512):
    model.eval()
    err = torch.cat([(model(X[i:i + bs].to(DEVICE)).cpu() - Y[i:i + bs]).abs() * S[i:i + bs, None]
                     for i in range(0, len(X), bs)])
    return {"mae_kw": err.mean().item(), "rmse_kw": err.pow(2).mean().sqrt().item()}


def run_one(arm, seed):
    set_seed(seed)
    t = CFG["train"]
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
        floor = torch.cuda.memory_allocated()
        torch.cuda.reset_peak_memory_stats()
    model = build_model(arm).to(DEVICE)
    n_params = sum(p.numel() for p in model.parameters())
    opt = torch.optim.AdamW(model.parameters(), lr=t["lr"], weight_decay=t["weight_decay"])
    Xtr, Ytr, _ = W["train"]
    steps = t["epochs"] * math.ceil(len(Xtr) / t["batch_size"])
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=t["lr"], total_steps=steps, pct_start=0.1)
    gen = torch.Generator().manual_seed(seed)
    rss0 = psutil.Process().memory_info().rss
    t0 = time.perf_counter()
    best, best_state, curve = 1e9, None, []
    for _ in range(t["epochs"]):
        model.train()
        perm = torch.randperm(len(Xtr), generator=gen)
        for i in range(0, len(perm), t["batch_size"]):
            idx = perm[i:i + t["batch_size"]]
            loss = F.l1_loss(model(Xtr[idx].to(DEVICE)), Ytr[idx].to(DEVICE))
            opt.zero_grad(set_to_none=True)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), t["grad_clip"])
            opt.step()
            sched.step()
        v = evaluate(model, *W["val"])["mae_kw"]
        curve.append(v)
        if v < best:  # same checkpoint rule for every arm
            best, best_state = v, {k: x.clone() for k, x in model.state_dict().items()}
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    train_s = time.perf_counter() - t0
    model.load_state_dict(best_state)
    te = evaluate(model, *W["test"])
    peak = (torch.cuda.max_memory_allocated() - floor) / 2**20 if DEVICE.type == "cuda" else float("nan")
    return {"arm": arm, "seed": seed, "n_params": n_params, "val_mae_kw": best, "test_mae_kw": te["mae_kw"],
            "test_rmse_kw": te["rmse_kw"], "best_epoch": int(np.argmin(curve)) + 1, "train_s": train_s,
            "peak_gpu_mb": peak, "rss_delta_mb": (psutil.Process().memory_info().rss - rss0) / 2**20}

## Step 7: The Ablation Loop

Seeds are the outer loop and the arm order inside each seed is shuffled with a generator seeded by the seed, so
machine drift (thermal throttling, other processes) lands on all arms alike instead of on whichever ran last.
`results/runs_<tag>.csv` is rewritten after every run and finished (arm, seed) pairs are skipped on a rerun.

References: [1](https://docs.pytorch.org/docs/stable/notes/randomness.html)

In [ ]:
RUNS_CSV = RESULTS / f"runs_{TAG}.csv"
rows = [] if FRESH or not RUNS_CSV.exists() else pd.read_csv(RUNS_CSV).to_dict("records")
done = {(r["arm"], int(r["seed"])) for r in rows}
for seed in SEEDS:
    order = list(np.random.default_rng(1000 + seed).permutation(ARMS))
    for arm in order:
        if (arm, seed) in done:
            continue
        r = run_one(arm, seed)
        rows.append(r)
        pd.DataFrame(rows).to_csv(RUNS_CSV, index=False)
        print(f"seed={seed} {arm:11s} test_mae={r['test_mae_kw']:.3f} kW  train={r['train_s']:.0f}s")
RUNS = pd.DataFrame(rows)

## Step 8: Latency Pass (Interleaved)

Latency is measured after training, not inside it. Each round visits every arm once, in a freshly shuffled
order; a visit is `warmup` discarded forwards then `repeats` timed forwards of a fixed batch, synchronised on
GPU. Latency depends on the architecture, not on trained weights, so freshly initialised models are timed. The
table reports the mean over rounds of each round's p50 and p95, and the spread between rounds.

References: [1](https://docs.pytorch.org/docs/stable/generated/torch.cuda.synchronize.html)

In [ ]:
def time_forward(model, xb):
    lc = CFG["latency"]
    out = []
    with torch.no_grad():
        for r in range(lc["warmup"] + lc["repeats"]):
            if DEVICE.type == "cuda":
                torch.cuda.synchronize()
            s = time.perf_counter()
            model(xb)
            if DEVICE.type == "cuda":
                torch.cuda.synchronize()
            if r >= lc["warmup"]:
                out.append((time.perf_counter() - s) * 1000)
    return np.percentile(out, 50), np.percentile(out, 95)


LAT_CSV = RESULTS / f"latency_runs_{TAG}.csv"
if LAT_CSV.exists() and not FRESH:  # a resume reuses the saved measurement, like it reuses saved training runs
    LAT = pd.read_csv(LAT_CSV)
    print(f"latency: reusing {LAT_CSV.name} (ABLATE_FRESH=1 measures again)")
else:
    set_seed(0)
    lat_models = {a: build_model(a).to(DEVICE).eval() for a in ARMS}
    lat_x = W["test"][0][:CFG["latency"]["batch_size"]].to(DEVICE)
    lat_rows = []
    rng_lat = np.random.default_rng(2024)
    for rnd in range(CFG["latency"]["rounds"]):
        for arm in rng_lat.permutation(ARMS):
            p50, p95 = time_forward(lat_models[arm], lat_x)
            lat_rows.append({"arm": arm, "round": rnd, "p50_ms_window": p50 / len(lat_x), "p95_ms_window": p95 / len(lat_x)})
    LAT = pd.DataFrame(lat_rows)
    LAT.to_csv(LAT_CSV, index=False)
print(LAT.groupby("arm")[["p50_ms_window", "p95_ms_window"]].mean().reindex(ARMS).round(4))

## Step 9: Statistics and Verdict

Per-arm mean, seed SD and 95% CI. Each arm is compared with the reference arm by a Welch test (primary: no
pairing assumption, because arms with different parameter shapes do not share initial weights), with Holm
correction across the comparisons. A paired p-value is shown as a secondary check. "Exceeds noise" means the
Holm-adjusted Welch p is below 0.05. With few seeds the test has low power: "not distinguishable" is not "equal".

References: [1](https://doi.org/10.1093/biomet/34.1-2.28) · [2](https://www.jstor.org/stable/4615733) · [3](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.ttest_ind.html)

In [ ]:
def ci95(x):
    x = np.asarray(x, float)
    return float("nan") if len(x) < 2 else stats.t.ppf(0.975, len(x) - 1) * x.std(ddof=1) / np.sqrt(len(x))


def holm(p):
    order, adj, running = np.argsort(p), np.empty(len(p)), 0.0
    for rank, i in enumerate(order):
        running = max(running, (len(p) - rank) * p[i])
        adj[i] = min(1.0, running)
    return adj


metric = "test_mae_kw"
g = RUNS.groupby("arm")
lat_mean = LAT.groupby("arm")[["p50_ms_window", "p95_ms_window"]].mean()
lat_spread = LAT.groupby("arm")["p50_ms_window"].std(ddof=1)
SUMM = pd.DataFrame({
    "n_seeds": g.size(), "n_params": g.n_params.first(),
    "mae_mean": g[metric].mean(), "mae_sd": g[metric].std(ddof=1), "mae_ci95_halfwidth": g[metric].apply(ci95),
    "mae_min": g[metric].min(), "mae_max": g[metric].max(), "rmse_mean": g.test_rmse_kw.mean(),
    "train_s_mean": g.train_s.mean(), "p50_ms_window": lat_mean.p50_ms_window, "p95_ms_window": lat_mean.p95_ms_window,
    "p50_round_sd": lat_spread,
    # median over seeds: the first CUDA run of a session pays one-time library workspace, so a max would charge it to one arm
    "peak_gpu_mb": g.peak_gpu_mb.median(), "rss_delta_mb_mean": g.rss_delta_mb.mean(),
}).reindex(ARMS)
SUMM.round(4).to_csv(RESULTS / f"summary_{TAG}.csv")

ref = RUNS[RUNS.arm == REF].set_index("seed")[metric]
cmp_rows = []
for arm in [a for a in ARMS if a != REF]:
    x = RUNS[RUNS.arm == arm].set_index("seed")[metric]
    d = x.values.mean() - ref.values.mean()
    vx, vr = x.var(ddof=1) / len(x), ref.var(ddof=1) / len(ref)
    se = np.sqrt(vx + vr)
    dfw = se**4 / (vx**2 / (len(x) - 1) + vr**2 / (len(ref) - 1))
    half = stats.t.ppf(0.975, dfw) * se
    common = x.index.intersection(ref.index)
    cmp_rows.append({
        "arm": arm, "reference": REF, "diff_mae_kw": d, "diff_pct_of_ref": 100 * d / ref.mean(),
        "welch_ci_lo": d - half, "welch_ci_hi": d + half,
        "welch_p": stats.ttest_ind(x, ref, equal_var=False).pvalue,
        "paired_p": stats.ttest_rel(x[common], ref[common]).pvalue,
        "effect_over_seed_sd": abs(d) / np.sqrt((x.var(ddof=1) + ref.var(ddof=1)) / 2)})
CMP = pd.DataFrame(cmp_rows)
CMP["welch_p_holm"] = holm(CMP.welch_p.values)
CMP["exceeds_noise"] = CMP.welch_p_holm < 0.05
CMP.round(5).to_csv(RESULTS / f"comparison_{TAG}.csv", index=False)

VERDICTS = []
for _, c in CMP.iterrows():
    if c.exceeds_noise:
        word = "better" if c.diff_mae_kw < 0 else "worse"
        VERDICTS.append(f"{c.arm} vs {REF}: EXCEEDS run-to-run noise ({c.diff_mae_kw:+.3f} kW, {c.diff_pct_of_ref:+.1f}%, "
                        f"{c.arm} {word}; Holm-adjusted p={c.welch_p_holm:.3g}; effect = {c.effect_over_seed_sd:.1f}x the seed SD).")
    else:
        VERDICTS.append(f"{c.arm} vs {REF}: NOT distinguishable from noise ({c.diff_mae_kw:+.3f} kW, {c.diff_pct_of_ref:+.1f}%; "
                        f"Welch 95% CI [{c.welch_ci_lo:+.3f}, {c.welch_ci_hi:+.3f}] includes 0; Holm-adjusted p={c.welch_p_holm:.3g}).")
for v in VERDICTS:
    print("-", v)

## Step 10: Tables and Plots

`results_<tag>.md` holds the per-arm table, the comparison table and the verdict. The first plot shows every
seed as a dot with the mean and 95% CI; the second shows each arm's difference from the reference with its Welch
interval; the third shows accuracy against latency.

References: [1](https://matplotlib.org/stable/gallery/statistics/errorbar_features.html)

In [ ]:
def md_table(df, digits=3, index=True):
    d = df.reset_index() if index else df
    cols = [str(c) for c in d.columns]
    fmt = lambda v: f"{v:.{digits}f}" if isinstance(v, (float, np.floating)) else str(v)
    return "\n".join(["| " + " | ".join(cols) + " |", "|" + "---|" * len(cols)]
                     + ["| " + " | ".join(fmt(v) for v in row) + " |" for row in d.itertuples(index=False)])


md = [f"# Results ({CFG['name']}, tag `{TAG}`)", "",
      f"Metric: test MAE in kW (lower is better), {int(SUMM.n_seeds.min())} seeds per arm. Reference arm: `{REF}`. "
      f"'Exceeds noise' = Welch p < 0.05 after Holm correction across {len(CMP)} comparisons.", "",
      "## Per-arm results", "", md_table(SUMM), "", "## Each arm vs reference", "", md_table(CMP, 4, index=False), "",
      f"No-learning references: same hours yesterday MAE = {NAIVE['naive_day_mae_kw']:.2f} kW; "
      f"same hours last week = {NAIVE['naive_week_mae_kw']:.2f} kW.", "", "## Verdict", ""]
md += [f"- {v}" for v in VERDICTS]
md += ["", f"Caveat: with {int(SUMM.n_seeds.min())} seeds per arm the test has low power; 'not distinguishable' means "
       "'not shown', not 'equal'. All seeds share one synthetic dataset, so the seed spread excludes data-sampling variance."]
(RESULTS / f"results_{TAG}.md").write_text("\n".join(md), encoding="utf-8")

colors = dict(zip(ARMS, ["#7a7a7a", "#1f77b4", "#d95f02", "#1b9e77"]))
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for i, a in enumerate(ARMS):
    v = RUNS[RUNS.arm == a][metric].values
    ax[0].scatter(np.full(len(v), i) + np.linspace(-.12, .12, len(v)), v, color=colors[a], alpha=.8, zorder=3)
    ax[0].errorbar(i, v.mean(), yerr=ci95(v), color="k", capsize=6, lw=2, zorder=4)
ax[0].set_xticks(range(len(ARMS)), ARMS)
ax[0].set_ylabel("test MAE (kW)")
ax[0].set_title("Every seed (dots), mean +/- 95% CI (bar)")
ax[0].axhline(NAIVE["naive_day_mae_kw"], ls="--", c="grey", lw=1)
ax[0].text(len(ARMS) - .6, NAIVE["naive_day_mae_kw"], "yesterday-naive", ha="right", va="bottom", fontsize=8, color="grey")
for a in ARMS:
    ax[1].scatter(SUMM.loc[a, "p50_ms_window"], SUMM.loc[a, "mae_mean"], color=colors[a], s=60, label=a)
ax[1].set_xlabel("inference latency (ms / window, p50 over interleaved rounds)")
ax[1].set_ylabel("mean test MAE (kW)")
ax[1].set_title("Accuracy vs cost")
ax[1].legend()
fig.tight_layout()
fig.savefig(RESULTS / f"mae_by_arm_{TAG}.png", dpi=150)
if IN_NOTEBOOK:
    plt.show()
plt.close(fig)

fig, ax = plt.subplots(figsize=(6, 3.5))
for i, c in CMP.reset_index(drop=True).iterrows():
    ax.errorbar(c.diff_mae_kw, i, xerr=[[c.diff_mae_kw - c.welch_ci_lo], [c.welch_ci_hi - c.diff_mae_kw]],
                fmt="o", color=colors[c.arm], capsize=5)
ax.axvline(0, c="k", lw=1)
ax.set_yticks(range(len(CMP)), CMP.arm)
ax.set_xlabel(f"MAE difference vs {REF} (kW), Welch 95% CI; <0 = better")
fig.tight_layout()
fig.savefig(RESULTS / f"diff_vs_reference_{TAG}.png", dpi=150)
if IN_NOTEBOOK:
    plt.show()
plt.close(fig)

## Step 11: Results JSON and Manifest

`results_<tag>.json` carries every number the README renders; `manifest_<tag>.json` records the hardware, library
versions, data hash and full config so another engineer can tell what the numbers were measured on.

References: [1](https://docs.python.org/3/library/json.html)

In [ ]:
total_s = time.perf_counter() - T_START
MANIFEST = RESULTS / f"manifest_{TAG}.json"
if MANIFEST.exists() and not FRESH:  # a resumed invocation adds to the runtime already recorded
    total_s += json.loads(MANIFEST.read_text(encoding="utf-8")).get("total_runtime_s", 0.0)
arms_json = {a: {k: (None if (isinstance(v, float) and math.isnan(v)) else (v.item() if hasattr(v, "item") else v))
                 for k, v in SUMM.loc[a].items()} for a in ARMS}
(RESULTS / f"results_{TAG}.json").write_text(json.dumps({
    "tag": TAG, "reference_arm": REF, "seeds": SEEDS, "arms": arms_json,
    "comparison": json.loads(CMP.to_json(orient="records")), "verdicts": VERDICTS, "naive": NAIVE,
    "reference_checks": CHECKS}, indent=1), encoding="utf-8")
MANIFEST.write_text(json.dumps({
    "hardware": hardware_snapshot(ENV), "total_runtime_s": total_s, "data_sha256": DATA_SHA,
    "cpu_threads_pinned": CFG["threads"], "quick": QUICK, "config": CFG}, indent=1, default=str), encoding="utf-8")
print(f"wrote results/*_{TAG}.* in {total_s:.0f}s total")

## Step 12: Power Analysis

Computes the smallest true difference the seed budget could detect, and the T4 time needed to resolve smaller ones, from the per-seed rows this session just wrote. Output: `results/power_<tag>.*`.

In [ ]:
import subprocess, sys
from IPython.display import Image, display

subprocess.run([sys.executable, "scripts/power_analysis.py", "--tag", ENV["tag"]], check=True)
display(Image("results/power_" + ENV["tag"] + ".png"))

## Step 13: Repeat the Whole Study

Retrains every (arm, seed) run from scratch `REPEATS` times and reports how far arm means, latency and training time move between repeats, and whether per-run test MAEs are bit-identical. Each repeat is archived under `results/repeats/<tag>/rep<k>/`; the single-session results above are restored afterwards. About 8 minutes per repeat on a T4. A dropped session resumes: finished repeats are skipped.

In [ ]:
REPEATS = 3  # the assignment standard is at least three
subprocess.run([sys.executable, "scripts/repeat_study.py", "--n", str(REPEATS)], check=True)

## Step 14: Write Results Back to GitHub

Asks before touching `README.md`, then asks again before pushing. The token comes from the Colab secret `GITHUB_TOKEN` (Secrets panel, with Notebook access on) or a hidden prompt. It is sent per command as an HTTP header and is never printed, stored in `.git/config`, or placed in the remote URL. See `COLAB.md`.

In [ ]:
sys.path.insert(0, "scripts")
from update_readme import offer_publish

offer_publish(ROOT, ENV["tag"], interactive=True)